# Prompt Chaining with Structured Output

The companion notebook `01_Prompt_Chaining.ipynb` in this folder demonstrates the classic version of prompt chaining: each step in the sequence hands the next step a raw string, and the next step's prompt simply interpolates that string into free text. That works, but it has a hidden cost — every step has to *trust* that the previous step's text is in a shape it can parse (a list here, a specific tone there), and there is no contract enforcing that.

This notebook shows the same workflow primitive with one change: **every step emits a validated Pydantic object instead of a raw string, and that typed object — not free text — is what feeds the next step's prompt.** This is the pattern popularized as the "JSON Example" style of prompt chaining (see e.g. `evoiz/Agentic-Design-Patterns`, Chapter 1): a chain of LLM calls that exchange structured, schema-validated data instead of prose.

### Definition
**Prompt Chaining with Structured Output** decomposes a complex task into a sequence of LLM calls, exactly like plain prompt chaining, but constrains every call's output to a Pydantic schema via `llm.with_structured_output(SomeModel)`. The parsed, validated object from step *N* is passed as typed fields into the prompt for step *N+1*, instead of being passed as an opaque blob of text.

### When to Use
*   **Multi-stage pipelines that feed a program, not just a human.** If step 2 needs to loop over a list that step 1 produced, or read a specific numeric field, you don't want to regex it out of prose.
*   **Long chains (3+ steps).** Formatting drift compounds: a free-text chain's outputs get subtly reformatted at every hop, and by step 4 the shape the last step needs may no longer be reliably present.
*   **Anywhere downstream code (not just the next LLM call) consumes the output** — e.g. writing to a database, rendering a UI, or calling another tool.

### Strengths & Weaknesses
*   **Strengths:**
    *   **Fewer parsing errors.** The schema is enforced by the model provider's structured-output machinery, not by hoping the model keeps formatting consistent turn after turn.
    *   **Self-documenting contracts.** The Pydantic model *is* the interface between steps — field names and descriptions tell you (and the LLM) exactly what each stage must produce.
    *   **Reduced hallucinated formatting.** The model can't decide to switch from a numbered list to bullet points to prose halfway through a chain; the shape is fixed.
*   **Weaknesses:**
    *   **Upfront schema design cost.** You have to think through the data model for every step before writing prompts.
    *   **Rigidity.** A schema that's too narrow can force the model to omit useful nuance that free text would have captured.
    *   **Provider/model support.** Structured output quality varies by model; smaller or older models may need retries or fall back to weaker JSON-mode parsing.

## Phase 0: Setup

**What we are going to do:**
We import `get_llm` from the shared `helpers` package (the repo-wide factory that picks the right provider/model for the current platform) along with `pydantic`, `typing`, and `langgraph` building blocks. We never instantiate `ChatOpenAI` / `ChatGroq` / `ChatDatabricks` directly — `get_llm()` handles that.

In [ ]:
# ============ SETUP: IMPORTS ============
from typing import List, Optional

from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END

from helpers import get_llm

print("Imports ready.")

In [ ]:
# ============ SETUP: INITIALIZE LLM ============
llm = get_llm()

if hasattr(llm, "model_name"):
    print(f"LLM initialized: {llm.model_name}")
elif hasattr(llm, "model"):
    print(f"LLM initialized: {llm.model}")
else:
    print("LLM initialized.")

## Phase 1: Defining the Schemas for Each Chain Step

**What we are going to do:**
We will chain three steps that take a one-sentence project idea all the way to an implementation plan:

1. **Extract Requirements** — raw idea → `ProjectRequirements`
2. **Generate Design** — `ProjectRequirements` → `SystemDesign`
3. **Generate Implementation Plan** — `SystemDesign` → `ImplementationPlan`

Each Pydantic model below is the *contract* for one step's output. Field descriptions matter here — they are shown to the model as part of the structured-output schema, so a clear description directly improves extraction quality.

In [ ]:
# ============ SCHEMA: STEP 1 OUTPUT ============
class ProjectRequirements(BaseModel):
    """Structured requirements extracted from a raw project idea."""

    project_name: str = Field(description="A short, descriptive name for the project")
    problem_statement: str = Field(description="The core problem this project solves")
    functional_requirements: List[str] = Field(
        description="Concrete, testable functional requirements (3-6 items)"
    )
    target_users: str = Field(description="Who will use this system")
    priority: str = Field(description="Overall priority: 'low', 'medium', or 'high'")


# ============ SCHEMA: STEP 2 OUTPUT ============
class SystemDesign(BaseModel):
    """Structured system design produced from a set of requirements."""

    architecture_style: str = Field(
        description="e.g. monolith, microservices, event-driven, serverless"
    )
    components: List[str] = Field(description="Major components/modules in the system")
    data_model_summary: str = Field(
        description="Brief description of the core data entities and how they relate"
    )
    tech_stack: List[str] = Field(description="Proposed technologies/frameworks")
    key_risks: List[str] = Field(
        description="Technical risks or open questions raised by this design"
    )


# ============ SCHEMA: STEP 3 OUTPUT ============
class ImplementationPlan(BaseModel):
    """Structured implementation plan produced from a system design."""

    milestones: List[str] = Field(description="Ordered list of implementation milestones")
    estimated_weeks: int = Field(description="Rough total estimate in weeks, as an integer")
    testing_strategy: str = Field(description="How the system will be tested")
    rollout_plan: str = Field(description="How the system will be deployed/released")


print("Schemas defined: ProjectRequirements -> SystemDesign -> ImplementationPlan")

### Step 1.1: Defining the Graph State

**What we are going to do:**
The chain's state is a `TypedDict` where each field after the first is one step's typed output. Notice the fields hold `Optional[ProjectRequirements]`, `Optional[SystemDesign]`, etc. — actual Pydantic model instances, not strings.

In [ ]:
class ChainState(TypedDict):
    """State threaded through the structured prompt chain."""

    project_idea: str
    requirements: Optional[ProjectRequirements]
    design: Optional[SystemDesign]
    plan: Optional[ImplementationPlan]


print("ChainState TypedDict defined.")

## Phase 2: Building the Structured Chain Steps

**What we are going to do:**
Each node below calls `llm.with_structured_output(SomeModel)` instead of a plain `llm.invoke(...)`. The return value is already a validated instance of `SomeModel` — there is no JSON string to parse, no markdown code fence to strip, and no risk of the model wrapping its answer in commentary. We then read typed fields (`req.project_name`, `design.components`, ...) directly off the previous step's object when building the next prompt.

In [ ]:
# ============ NODE 1: EXTRACT REQUIREMENTS ============
def extract_requirements(state: ChainState) -> dict:
    """Step 1: turn a raw idea into structured requirements."""
    structured_llm = llm.with_structured_output(ProjectRequirements)

    prompt = f"""Analyze this project idea and extract structured requirements from it.

Project idea: {state['project_idea']}

Be specific and concrete. Do not invent details that aren't implied by the idea."""

    result = structured_llm.invoke(prompt)
    print(f"✅ Requirements extracted for: {result.project_name}")
    return {"requirements": result}

In [ ]:
# ============ NODE 2: GENERATE DESIGN ============
def generate_design(state: ChainState) -> dict:
    """Step 2: turn structured requirements into a structured system design."""
    structured_llm = llm.with_structured_output(SystemDesign)
    req = state["requirements"]

    prompt = f"""Design a system architecture that satisfies these requirements.

Project name: {req.project_name}
Problem statement: {req.problem_statement}
Target users: {req.target_users}
Priority: {req.priority}
Functional requirements:
{chr(10).join(f'- {r}' for r in req.functional_requirements)}

Propose a concrete, implementable architecture."""

    result = structured_llm.invoke(prompt)
    print(
        f"✅ Design proposed: {result.architecture_style} "
        f"with {len(result.components)} components"
    )
    return {"design": result}

In [ ]:
# ============ NODE 3: GENERATE IMPLEMENTATION PLAN ============
def generate_plan(state: ChainState) -> dict:
    """Step 3: turn a structured system design into a structured implementation plan."""
    structured_llm = llm.with_structured_output(ImplementationPlan)
    design = state["design"]

    prompt = f"""Create a realistic implementation plan for this system design.

Architecture style: {design.architecture_style}
Data model summary: {design.data_model_summary}
Tech stack: {', '.join(design.tech_stack)}
Components:
{chr(10).join(f'- {c}' for c in design.components)}
Known risks:
{chr(10).join(f'- {r}' for r in design.key_risks)}

Produce an ordered set of milestones with a total time estimate."""

    result = structured_llm.invoke(prompt)
    print(
        f"✅ Plan generated: {len(result.milestones)} milestones, "
        f"~{result.estimated_weeks} weeks"
    )
    return {"plan": result}

**Discussion of the Output:**
Notice what's absent from these node functions compared to a free-text chain: there is no string splitting, no regex, no "hope the model used a numbered list." `req.functional_requirements` is already a Python `list[str]`; `design.components` is already iterable. The typed contract from one step becomes directly usable Python in the next.

## Phase 3: Assembling the Chain

**What we are going to do:**
We wire the three nodes into a strictly sequential `StateGraph`, identical in shape to the plain-text chain in `01_Prompt_Chaining.ipynb` — the difference is entirely in what flows along the edges (validated objects, not strings).

In [ ]:
# ============ BUILD THE SEQUENTIAL GRAPH ============
workflow = StateGraph(ChainState)

workflow.add_node("extract_requirements", extract_requirements)
workflow.add_node("generate_design", generate_design)
workflow.add_node("generate_plan", generate_plan)

workflow.add_edge(START, "extract_requirements")
workflow.add_edge("extract_requirements", "generate_design")
workflow.add_edge("generate_design", "generate_plan")
workflow.add_edge("generate_plan", END)

structured_chain = workflow.compile()

print("Structured prompt chain compiled successfully!")

In [ ]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

display(
    Image(
        structured_chain.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

## Phase 4: End-to-End Execution

**What we are going to do:**
We invoke the chain with a single-sentence project idea and print each step's validated Pydantic object as pretty JSON via `.model_dump_json(indent=2)`. Because every intermediate object is schema-validated, we can serialize, log, or persist any stage of the chain without writing a custom parser.

In [ ]:
initial_state: ChainState = {
    "project_idea": (
        "A tool that helps small businesses manage inventory across multiple "
        "warehouses using barcode scanning."
    ),
    "requirements": None,
    "design": None,
    "plan": None,
}

print(f"🚀 Running structured chain for: '{initial_state['project_idea']}'")
print("-" * 60)

final_state = structured_chain.invoke(initial_state)

print("\n=== REQUIREMENTS ===")
print(final_state["requirements"].model_dump_json(indent=2))

print("\n=== SYSTEM DESIGN ===")
print(final_state["design"].model_dump_json(indent=2))

print("\n=== IMPLEMENTATION PLAN ===")
print(final_state["plan"].model_dump_json(indent=2))

**Discussion of the Output:**
Each block above is a validated Pydantic object dumped to JSON — not a model's best attempt at formatting prose into something JSON-shaped after the fact. Compare this to the free-text chain in `01_Prompt_Chaining.ipynb`, where `validate_key_points` has to reach for keyword-counting heuristics on raw text to decide whether the previous step's output was "good enough" to proceed. Here, `ProjectRequirements`, `SystemDesign`, and `ImplementationPlan` are guaranteed — by the structured-output call itself — to have every required field, of the right type, before the next node ever runs. Validation moves from being a downstream, best-effort check to being a precondition enforced at each hop.

## Key Takeaways

*   **Prompt chaining with structured output is the same workflow primitive as plain prompt chaining** — decompose a task into sequential LLM calls — but each step's output is a schema-validated Pydantic object instead of raw text.
*   **`llm.with_structured_output(SomeModel)` replaces `llm.invoke(...)` at every step**, and the returned object's typed fields are read directly when building the next step's prompt — no regex, no manual JSON parsing, no brittle text-splitting.
*   **This reduces two classes of failure that plague free-text chains:** parsing errors (the next step can't find the data it needs) and hallucinated/drifting formatting (the model changes its output shape mid-chain).
*   **The tradeoff is upfront schema design.** You must think through each step's data contract (the Pydantic model) before writing the prompt — but that contract then documents itself and is directly reusable by downstream code (logging, persistence, UI rendering) with zero extra glue.
*   **Use plain-text chaining (see `01_Prompt_Chaining.ipynb`) when the output is meant for a human to read as-is**; reach for structured-output chaining whenever a later step, or any non-LLM code, needs to consume specific fields reliably.